In [1]:
import os

import torch
from torch import Tensor

import torch_geometric
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter, add_self_loops, degree
from torch_geometric.nn import GCNConv

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from tqdm import tqdm, trange

In [2]:
# CONSTANTS 
DOMAIN_SIZE = 1000
DATA_FOLDER = os.path.join('data', 'boids')
RAW_DATA_FOLDER = os.path.join(DATA_FOLDER, 'raw', '')
PROCESSED_DATA_FOLDER = os.path.join(DATA_FOLDER, 'processed', '')

In [3]:
def square_torus_distance(p1, p2, width=1.0, height=1.0):
    """
    Compute the squared torus distance between two points p1 and p2 in a box of given width and height.

    Args:
        p1: Tensor of shape (num_points, 2) representing the first point(s)
        p2: Tensor of shape (num_points, 2) representing the second point(s)
        width: Width of the box
        height: Height of the box
    Returns:
        dist: Tensor of shape (num_points,) representing the squared torus distance between p1 and p2
    """
    dx = p1[:, 0] - p2[:, 0]
    dy = p1[:, 1] - p2[:, 1]
    dx = dx - width * torch.round(dx / width)
    dy = dy - height * torch.round(dy / height)
    return dx ** 2 + dy ** 2

def pbc_direction(p1: Tensor, p2: Tensor) -> Tensor:
    """
    Compute the direction from p1 to p2 considering periodic boundary conditions in a unit square.
    Args:
        p1: Tensor of shape (num_points, 2) representing the first point(s)
        p2: Tensor of shape (num_points, 2) representing the second point(s)
    Returns:
        direction: Tensor of shape (num_points, 2) representing the direction from p1 to p2 considering PBCs
    """
    return p2 - p1 - torch.round(p2 - p1)


In [4]:
class AR_Boids_Dataset(InMemoryDataset):
    def __init__(self, raw_data_path, processed_data_path, root=None, transform=None, pre_transform=None, post_transform=None, solution_idx_range=(0, 25), timesteps=1000, processed_file_name="AR3_Boids.pt"):
        self.raw_data_path = raw_data_path
        self.processed_data_path = processed_data_path
        self.solution_idx_range = solution_idx_range
        self.timesteps = timesteps
        self.processed_file_name = processed_file_name
        self.pre_transform = pre_transform
        self.transform = transform
        self.post_transform = post_transform
        super(AR_Boids_Dataset, self).__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)

    @property
    def processed_file_names(self):
        return [self.processed_file_name]

    @property
    def raw_file_names(self):
        return [pfn for pfn in os.listdir(self.raw_data_path) if (self.solution_idx_range[0] <= int(pfn.split("_")[-1][:-4]) < self.solution_idx_range[1])]
    
    def download(self):
        pass
    
    def __len__(self):
        return (self.timesteps - 1) * (self.solution_idx_range[1] - self.solution_idx_range[0])

    def process(self):
        positions_list = []
        data_list = []
        for idx, raw_path in enumerate(self.raw_file_names):
            trajectory = np.load(self.raw_data_path + raw_path)

            if self.transform is not None:
                trajectory = self.transform(trajectory)
                
            # Add the initial positions to the positions list
            positions_list.append(trajectory[0, :, :2])

            for t in trange(trajectory.shape[0] - 1):
                x = torch.tensor(trajectory[t], dtype=torch.float)
                y = torch.tensor(trajectory[t+1], dtype=torch.float)
                
                # Create fully connected graph
                n = trajectory.shape[1]
                edge_index = torch.tensor([[i, j] for i in range(n) for j in range(n) if i != j], dtype=torch.long).t().contiguous()
                
                data = Data(x=x, y=y, edge_index=edge_index)
                if self.post_transform is not None:
                    data = self.post_transform(data)
                
                data_list.append(data)
                
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_data_path+self.processed_file_name)
        torch.save(torch.tensor(positions_list), self.processed_data_path+"positions_"+self.processed_file_name)

    def __getitem__(self, idx):
        return self.get(idx)
    
    def __repr__(self):
        return f'{self.__class__.__name__}({len(self)})'

In [5]:
train_dataset = AR_Boids_Dataset(
    raw_data_path=RAW_DATA_FOLDER, 
    processed_data_path=PROCESSED_DATA_FOLDER, 
    root=DATA_FOLDER, 
    solution_idx_range=(0, 15), 
    timesteps=1000, 
    processed_file_name="AR3_Boids_Equivariant.pt",
    transform=lambda traj: traj / DOMAIN_SIZE
)

validation_dataset = AR_Boids_Dataset(
    raw_data_path=RAW_DATA_FOLDER, 
    processed_data_path=PROCESSED_DATA_FOLDER, 
    root=DATA_FOLDER, 
    solution_idx_range=(16, 25), 
    timesteps=1000, 
    processed_file_name="AR3_VAL_Boids_Equivariant.pt",
    transform=lambda traj: traj / DOMAIN_SIZE
)

In [6]:
class GeometricFlowMatchingModel(torch.nn.Module):
    def __init__(self):
        super(GeometricFlowMatchingModel, self).__init__()

    def forward(self, t: Tensor, data_t: Data, data_c: Data) -> Tensor:
        raise NotImplementedError("This method should be implemented in a subclass")

    def generate(self, data_c: Data, data_0: Data, n_euler_steps: int, t_start=0.0, t_end=1.0):
        time_steps = torch.linspace(t_start, t_end, n_euler_steps + 1).to(data_0.x.device)
        data_t = data_0.clone()

        for i in range(n_euler_steps):
            data_t.x += (time_steps[i+1] - time_steps[i]) * self(t=time_steps[i].unsqueeze(-1), data_t=data_t, data_c=data_c)
            data_t.x[:, :2] %= 1.0

        return data_t.x

In [33]:
class EGNNLayer(torch.nn.Module):
    def __init__(self, h_dim=16, m_dim=16, hidden_dim=16):
        super(EGNNLayer, self).__init__()

        # Constants
        self.edge_dim = 3
        self.width = 1.0
        self.height = 1.0
        STD = 1e-5

        # edge messages
        self.phi_e = torch.nn.Sequential(
            torch.nn.Linear(h_dim * 2 + self.edge_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, m_dim),
            torch.nn.SiLU()
        )
        torch.nn.init.normal_(self.phi_e[2].weight, mean=0, std=STD)

        # speed
        self.phi_v = torch.nn.Sequential(
            torch.nn.Linear(m_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, 1)
        )
        torch.nn.init.normal_(self.phi_v[2].weight, mean=0, std=STD)

        # force
        phi_x_last_layer = torch.nn.Linear(hidden_dim, 1, bias=False)
        torch.nn.init.xavier_uniform_(phi_x_last_layer.weight, gain=0.001)
        self.phi_x = torch.nn.Sequential(
            torch.nn.Linear(m_dim, hidden_dim),
            torch.nn.SiLU(),
            phi_x_last_layer
        )

        # new hidden state
        self.node_embedding_nn = torch.nn.Sequential(
            torch.nn.Linear(h_dim + m_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, h_dim)
        )
        torch.nn.init.normal_(self.node_embedding_nn[2].weight, mean=0, std=STD)
    
    def phi_h(self, h, m):
        agg = torch.cat([h, m], dim=1)
        out = self.node_embedding_nn(agg)
        return out + h

    def forward(self, x, edge_index, edge_attr, h, v_init):
        # Calculate edge features
        mij = self.phi_e(torch.cat([h[edge_index[0]], h[edge_index[1]], edge_attr], dim=1))

        # Add repulsion/attraction
        x[:, 2:] = self.phi_v(h) * v_init + scatter(
            pbc_direction(x[edge_index[0], :2], x[edge_index[1], :2]) * self.phi_x(mij), 
            edge_index[0], 
            0, 
            dim_size=x.shape[0], 
            reduce='sum'
        )

        # Update position
        x[:, :2] += x[:, 2:]
        x[:, 0] %= self.width
        x[:, 1] %= self.height
        
        # Update h
        h = self.phi_h(h, scatter(mij, edge_index[0], 0, dim_size=h.size(0), reduce='sum'))

        return h, x
    
class FlowEGNNModel(GeometricFlowMatchingModel):
    def __init__(self, h_dim=16, m_dim=16, hidden_dim=16, layers=1):
        super(FlowEGNNModel, self).__init__()
        self.h_dim = h_dim
        self.embedding = torch.nn.Linear(4, h_dim)
        self.layers = torch.nn.ModuleList([EGNNLayer(h_dim, m_dim, hidden_dim) for _ in range(layers)])

    def calculate_edge_attributes(self, x, edge_index):
        edge_attr = torch.zeros((edge_index.shape[1], 3), dtype=torch.float, device=x.device)

        # Square torus distance (to get distance between boids with PBCs)
        edge_attr[:, 0] = square_torus_distance(x[edge_index[0], :2], x[edge_index[1], :2])

        # Cosine similarity of velocities (to get alignment of velocities)
        edge_attr[:, 1] = torch.nn.functional.cosine_similarity(x[edge_index[0], 2:], x[edge_index[1], 2:], dim=1)

        # Calculate l2 norm between velocities (to get speed difference)
        edge_attr[:, 2] = (torch.norm(x[edge_index[1], 2:], dim=1) - torch.norm(x[edge_index[0], 2:], dim=1)).pow(2)

        return edge_attr

    def forward(self, t: Tensor, data_t: Data, data_c: Data):
        x, edge_index = data_t.x.clone().detach(), data_t.edge_index
        x_init = x.clone()

        # Restrict edges
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.shape[0])
        edge_weights = square_torus_distance(x[edge_index[0], :2], x[edge_index[1], :2])
        edge_mask = edge_weights <= 0.04 ** 2
        edge_index = edge_index[:, edge_mask]
        edge_attr = self.calculate_edge_attributes(x, edge_index)

        # Create hidden layer
        # h = self.time_encoding(t.expand(x.shape[:-1]).unsqueeze(-1))
        h = torch.cat((t.expand(x.shape[:-1]).unsqueeze(-1), x[:, 2:], degree(edge_index[0], num_nodes=x.shape[0]).view(-1, 1)), dim=1)
        h = self.embedding(h)

        # Message passing
        for mp_layer in self.layers:
            h, x = mp_layer(x, edge_index, edge_attr, h, x_init[:, 2:])

        # Create output
        out = torch.zeros_like(x)
        out[:, :2] = pbc_direction(x_init[:, :2], x[:, :2])
        out[:, 2:] = x[:, 2:] - x_init[:, 2:]

        return out


In [46]:
def boids_path_sampler(t, x_1, x_c, sigma):
    # Initialize x_t and x_dot_t
    x_t = torch.zeros_like(x_1)
    x_dot_t = torch.zeros_like(x_1)
    
    # Sample standard normal noise
    noise = torch.randn_like(x_1)
    sigma_x, sigma_v = sigma

    # Compute x_t and x_dot_t
    x_t[:, :2] = (x_c[:, :2] + pbc_direction(x_c[:, :2], x_1[:, :2]) * t + sigma_x * noise[:, :2]) % 1.0
    x_t[:, 2:] = x_c[:, 2:] + (x_1[:, 2:] - x_c[:, 2:]) * t + sigma_v * noise[:, 2:]
    x_dot_t[:, :2] = pbc_direction(x_c[:, :2], x_1[:, :2])
    x_dot_t[:, 2:] = (x_1[:, 2:] - x_c[:, 2:])
    
    return x_t, x_dot_t

def ICFM_Boids_training(vf: GeometricFlowMatchingModel, dataset: AR_Boids_Dataset, n_epochs: int, sigma: tuple[float, float], device='cuda', loss_fn=torch.nn.MSELoss(), lr=1e-2):
    optimizer = torch.optim.Adam(vf.parameters(), lr)
    loss_hist = []
    N = len(dataset)
    for epoch in range(n_epochs):
        data_sample_n = 0
        random_indices = torch.randperm(N)
        for idx in tqdm(random_indices, leave=False):
            data = dataset[idx]
            data_c = data.clone().to(device)
            data_t = data.clone().to(device)

            x_c = data_t.x
            x_1 = data_t.y
            t = torch.rand(1).to(device)

            x_t, u = boids_path_sampler(t, x_1, x_c, sigma)

            data_t.x = x_t
            
            optimizer.zero_grad()
            loss = loss_fn(vf(t=t, data_t=data_t, data_c=data_c), u)
            loss.backward()
            optimizer.step()

            # loss = loss_fn(vf(t=t, data_t=data_t, data_c=data_c), u)
            # loss.backward()
            # if data_sample_n % 10 == 9:
            #     optimizer.step()
            #     optimizer.zero_grad()
            # data_sample_n += 1
        print('epoch: ', epoch, ', loss: ', loss.item())
        loss_hist.append(loss.item())    

    return loss_hist

In [49]:
flow_model = FlowEGNNModel(layers=2).to('cuda')
# sigma = (0.0, 0.0)
sigma = (5e-3, 5e-5)
loss_hist_ICFM = ICFM_Boids_training(flow_model, train_dataset, n_epochs=10, sigma=sigma, lr=1e-5)

epoch:  0 , loss:  3.751305044374931e-08


epoch:  1 , loss:  9.019803570708973e-09


epoch:  2 , loss:  7.0310361977021785e-09


epoch:  3 , loss:  1.4826257999089876e-09


epoch:  4 , loss:  1.2944401994730015e-08


epoch:  5 , loss:  1.8165782211809756e-09


epoch:  6 , loss:  2.238468743698263e-09


epoch:  7 , loss:  1.2992366071884476e-09


epoch:  8 , loss:  5.925481882940176e-09


epoch:  9 , loss:  1.397798876645595e-09


In [50]:
torch.save(flow_model.state_dict(), f"FM_EGNN.pt")

In [51]:
def keep_01(data):
    return data[0:2, :, :] / DOMAIN_SIZE

initial_states_validation_dataset = AR_Boids_Dataset(
    raw_data_path=RAW_DATA_FOLDER, 
    processed_data_path=PROCESSED_DATA_FOLDER, 
    root=DATA_FOLDER, 
    solution_idx_range=(16, 25), 
    timesteps=2, 
    processed_file_name="AR1_VAL_init.pt",
    transform=keep_01
)

In [52]:
@torch.no_grad()
def boids_rollout(model: GeometricFlowMatchingModel, init_data: Data, timesteps=1000, device='cuda', mode="residual", sigma=(0.0, 0.0)):
    """
    Predict the rollouts of the model on the dataset starting from the initial state

    Args:
        model: PyTorch model
        dataset: PyTorch dataset
        timesteps: Number of timesteps to predict
        device: Device to run the model on
        mode: "residual" or "direct"
        - In the solution above, we used the "residual" mode, where the model predicts the change in position and velocity
        - In the "direct" mode, the model predicts the position and velocity directly (if you do not intend to use this mode, you can ignore this argument)
        width: Width of the PBC box
        height: Height of the PBC box
    Returns:
        rollouts: Rollouts of the model on the dataset
        - Should be a torch tensor of shape (Batch, Timesteps, Boids, Node_dim)
    """
    # Allocate memory for the rollouts
    rollouts = torch.empty((timesteps, *init_data.x.shape), device=device)

    # Get the initial state
    data_0 = init_data.clone().to(device)
    data_c = init_data.clone().to(device)
    current_position = init_data.x.clone().to(device)

    # Get the randomness
    sigma_x, sigma_v = sigma

    for t in trange(timesteps):
        data_0.x[:, :2] += sigma_x * torch.randn_like(data_0.x[:, :2])
        data_0.x[:, 2:] += sigma_v * torch.randn_like(data_0.x[:, 2:])
        out_data = model.generate(data_c=data_c, data_0=data_0, n_euler_steps=10)

        # Update
        if mode == "direct":
            current_position = out_data
        elif mode == "residual":
            current_position += out_data
        
        # Wrap around the positions
        current_position[:, 0] = current_position[:, 0] % 1.0
        current_position[:, 1] = current_position[:, 1] % 1.0

        # Update the model data
        data_0.x = current_position
        data_c.x = current_position

        # Store the current state
        rollouts[t] = data_c.x.clone().cpu()

    return rollouts


In [53]:
flow_model.eval()
data_sample = initial_states_validation_dataset[0].clone().to('cuda')
rollout = boids_rollout(flow_model, data_sample, timesteps=100, device='cuda', mode="direct", sigma=sigma)

print(rollout.shape)
print(rollout[-1])

100%|██████████| 100/100 [00:05<00:00, 17.53it/s]

torch.Size([100, 25, 4])
tensor([[ 0.6606,  0.4023, -0.0044, -0.0029],
        [ 0.6842,  0.6301, -0.0042,  0.0021],
        [ 0.5176,  0.0177, -0.0027,  0.0038],
        [ 0.5923,  0.9281,  0.0018,  0.0035],
        [ 0.3369,  0.6843,  0.0046,  0.0037],
        [ 0.5140,  0.3974,  0.0042,  0.0051],
        [ 0.7475,  0.4843, -0.0035, -0.0034],
        [ 0.9830,  0.1383, -0.0042, -0.0025],
        [ 0.7026,  0.0525, -0.0025,  0.0032],
        [ 0.9740,  0.5556,  0.0028, -0.0041],
        [ 0.2498,  0.7513, -0.0041,  0.0042],
        [ 0.6479,  0.4120, -0.0044, -0.0031],
        [ 0.7949,  0.2142,  0.0034,  0.0030],
        [ 0.8418,  0.1317,  0.0033,  0.0047],
        [ 0.3559,  0.2877, -0.0039,  0.0045],
        [ 0.7048,  0.3818,  0.0022, -0.0049],
        [ 0.8299,  0.8977,  0.0033, -0.0024],
        [ 0.5828,  0.3552, -0.0034, -0.0031],
        [ 0.8096,  0.1007,  0.0031, -0.0023],
        [ 0.1466,  0.1639,  0.0030, -0.0019],
        [ 0.2096,  0.0567, -0.0038,  0.0038],
        [

In [54]:
def animate_rollout(rollouts, output_path="output/rollout.gif", width = 1.0, height = 1.0, max_timesteps=100):
    # rollouts of shape (Timesteps, Boids, Node_dim)
    
    # Create output directory if it does not exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Parse rollouts
    timesteps, num_boids, node_dim = rollouts.shape
    rollouts = rollouts.cpu().numpy()

    # Initialize the figure and axis
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(1, 1, 1)

    quiv = ax.quiver(rollouts[0, :, 0], rollouts[0, :, 1], rollouts[0, :, 2], rollouts[0, :, 3])
    scat = ax.scatter(rollouts[0, :, 0], rollouts[0, :, 1])
    ax.set_xlim(0, width)
    ax.set_ylim(0, height)
    ax.set_aspect('equal', adjustable='box')

    def update(frame):
        quiv.set_offsets(rollouts[frame, :, :2])
        quiv.set_UVC(rollouts[frame, :, 2], rollouts[frame, :, 3])
        scat.set_offsets(rollouts[frame, :, :2])
        return scat, quiv

    ani = animation.FuncAnimation(fig, update, frames=min(timesteps, max_timesteps), blit=False, interval=150)
    ani.save(output_path, writer="ffmpeg")
    plt.close()

In [56]:
animate_rollout(rollout, output_path="output/4.mp4")

In [61]:
# # Save a true trajectory
# for i in trange(26):
#     trajectory = np.load(os.path.join(RAW_DATA_FOLDER, f'boids_trajectory_{i}.npy')) / DOMAIN_SIZE
#     trajectory = torch.tensor(trajectory)
#     animate_rollout(trajectory, output_path=f"ground-output/{i}.gif", max_timesteps=1000)